# Joint Gini sweep

Combines the original sweep with the low-λ refinement for both regimes.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 320)

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError("Could not find repository root.")

REPO_ROOT = find_repo_root()
RESULTS_ROOT = REPO_ROOT / "opinion_dynamics" / "experiments" / "results"

SANITY_DIR = RESULTS_ROOT / "final_gini_sanity_analysis_2026_09_28"

SS_OLD_DIR = RESULTS_ROOT / "experiment_2026_09_29_single_shot_gini_joint_sweep"
AB_OLD_DIR = RESULTS_ROOT / "experiment_2026_09_29_abundant_gini_joint_sweep"

SS_LOW_DIR = RESULTS_ROOT / "experiment_2026_09_29_single_shot_gini_joint_refinement_low_sweep"
AB_LOW_DIR = RESULTS_ROOT / "experiment_2026_09_29_abundant_gini_joint_refinement_low_sweep"

OUT_DIR = RESULTS_ROOT / "joint_gini_sweep_with_low_refinement_2026_09_29"
OUT_DIR.mkdir(parents=True, exist_ok=True)

paths = {
    "anchor": SANITY_DIR / "gini_vs_unregularized_seed_level.csv",
    "ss_old": SS_OLD_DIR / "single_shot_gini_sweep_seed0.csv",
    "ab_old": AB_OLD_DIR / "abundant_gini_joint_sweep_seed0.csv",
    "ss_low": SS_LOW_DIR / "single_shot_gini_sweep_seed0.csv",
    "ab_low": AB_LOW_DIR / "abundant_gini_joint_sweep_seed0.csv",
}

for name, p in paths.items():
    if not p.exists():
        raise FileNotFoundError(f"{name}: {p}")

print("Repo:", REPO_ROOT)


Repo: d:\Work\repos\RL\unknown_graph_networks


C:\Users\Chainsword\AppData\Local\Temp\ipykernel_82900\2303099285.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
METRICS = [
    "control_final",
    "prediction_ratio",
    "A_MAE",
    "A_Fro",
    "row_TV",
    "true_edge_mass",
    "support_topdegree_recall",
    "edge_AP",
    "centrality_L1",
]

KEYS = [
    "regime",
    "environment_id",
    "scenario_class",
    "family",
    "dynamics",
    "topology_seed",
    "opinion_seed",
    "environment_seed",
    "learning_seed",
]

anchor = pd.read_csv(paths["anchor"])
anchor = anchor[anchor["learning_seed"] == 0].copy()
assert len(anchor) == 390

anchor_rows = []
for lam, suffix in [(0.0, "old"), (1e-4, "new")]:
    block = anchor[KEYS].copy()
    block["regularizer_lambda"] = lam
    for metric in METRICS:
        block[metric] = anchor[f"{metric}_{suffix}"].to_numpy()
    anchor_rows.append(block)
anchors = pd.concat(anchor_rows, ignore_index=True)

def load_ss(path):
    df = pd.read_csv(path)
    out = df[
        [
            "environment_id", "scenario_class", "family", "dynamics",
            "topology_seed", "opinion_seed", "environment_seed",
            "learning_seed", "regularizer_lambda",
            "final_mean", "final_fit_model_over_identity",
            "A_MAE", "A_Fro", "row_TV", "true_edge_mass",
            "support_topdegree_recall", "edge_AP", "centrality_L1",
        ]
    ].copy()
    out.insert(0, "regime", "single_shot")
    return out.rename(columns={
        "final_mean": "control_final",
        "final_fit_model_over_identity": "prediction_ratio",
    })

def load_ab(path):
    df = pd.read_csv(path)
    out = df[
        [
            "environment_id", "scenario_class", "family", "dynamics",
            "topology_seed", "opinion_seed", "environment_seed",
            "learning_seed", "regularizer_lambda",
            "mean_end", "model_over_identity",
            "A_MAE", "A_Fro", "row_TV", "true_edge_mass",
            "support_topdegree_recall", "edge_AP", "centrality_L1",
        ]
    ].copy()
    out.insert(0, "regime", "abundant")
    return out.rename(columns={
        "mean_end": "control_final",
        "model_over_identity": "prediction_ratio",
    })

work = pd.concat(
    [
        anchors,
        load_ss(paths["ss_old"]),
        load_ss(paths["ss_low"]),
        load_ab(paths["ab_old"]),
        load_ab(paths["ab_low"]),
    ],
    ignore_index=True,
    sort=False,
)

expected_lambdas = {
    0.0, 1e-7, 3e-7, 1e-6, 3e-6, 1e-5, 3e-5, 1e-4
}

assert set(work["regularizer_lambda"].astype(float).unique()) == expected_lambdas

counts = work.groupby(["regime", "regularizer_lambda"]).size()
assert counts.eq(195).all(), counts[counts != 195]

print("Validated:", len(expected_lambdas), "lambda values x 2 regimes x 195 environments")
display(counts.rename("n").reset_index())


Validated: 8 lambda values x 2 regimes x 195 environments


,regime,regularizer_lambda,n
0,abundant,0.000000e+00,195
1,abundant,1.000000e-07,195
2,abundant,3.000000e-07,195
3,abundant,1.000000e-06,195
4,abundant,3.000000e-06,195
5,abundant,1.000000e-05,195
6,abundant,3.000000e-05,195
7,abundant,1.000000e-04,195
8,single_shot,0.000000e+00,195
9,single_shot,1.000000e-07,195


In [3]:
ref = (
    work[work["regularizer_lambda"] == 0.0]
    .set_index(["regime", "environment_id"])
)

for metric in METRICS:
    work[f"delta_{metric}"] = work.apply(
        lambda r: float(r[metric]) - float(
            ref.loc[(r["regime"], r["environment_id"]), metric]
        ),
        axis=1,
    )

summary = (
    work.groupby(["regime", "regularizer_lambda"], as_index=False)
    .agg(
        n=("environment_id", "size"),
        control=("control_final", "mean"),
        delta_control=("delta_control_final", "mean"),
        prediction=("prediction_ratio", "mean"),
        delta_prediction=("delta_prediction_ratio", "mean"),
        row_TV=("row_TV", "mean"),
        delta_row_TV=("delta_row_TV", "mean"),
        true_edge_mass=("true_edge_mass", "mean"),
        delta_true_edge_mass=("delta_true_edge_mass", "mean"),
        edge_AP=("edge_AP", "mean"),
        delta_edge_AP=("delta_edge_AP", "mean"),
        centrality_L1=("centrality_L1", "mean"),
        delta_centrality_L1=("delta_centrality_L1", "mean"),
    )
)

by_dyn = (
    work.groupby(
        ["regime", "regularizer_lambda", "scenario_class", "dynamics"],
        as_index=False,
    )
    .agg(
        n=("environment_id", "size"),
        control=("control_final", "mean"),
        delta_control=("delta_control_final", "mean"),
        prediction=("prediction_ratio", "mean"),
        delta_prediction=("delta_prediction_ratio", "mean"),
        row_TV=("row_TV", "mean"),
        delta_row_TV=("delta_row_TV", "mean"),
        true_edge_mass=("true_edge_mass", "mean"),
        delta_true_edge_mass=("delta_true_edge_mass", "mean"),
        edge_AP=("edge_AP", "mean"),
        delta_edge_AP=("delta_edge_AP", "mean"),
        centrality_L1=("centrality_L1", "mean"),
        delta_centrality_L1=("delta_centrality_L1", "mean"),
    )
)

worst = (
    by_dyn.groupby(["regime", "regularizer_lambda"], as_index=False)
    .agg(
        worst_dynamic_control_delta=("delta_control", "min"),
        worst_dynamic_prediction_delta=("delta_prediction", "max"),
        worst_dynamic_row_TV_delta=("delta_row_TV", "max"),
    )
)

summary = summary.merge(
    worst,
    on=["regime", "regularizer_lambda"],
    validate="one_to_one",
)

summary.to_csv(OUT_DIR / "joint_gini_summary.csv", index=False)
by_dyn.to_csv(OUT_DIR / "joint_gini_by_dynamics.csv", index=False)

display(summary.round(6))


,regime,regularizer_lambda,n,control,delta_control,prediction,delta_prediction,row_TV,delta_row_TV,true_edge_mass,delta_true_edge_mass,edge_AP,delta_edge_AP,centrality_L1,delta_centrality_L1,worst_dynamic_control_delta,worst_dynamic_prediction_delta,worst_dynamic_row_TV_delta
0,abundant,0.000000,195,0.768628,0.000000,0.638310,0.000000,0.575996,0.000000,0.433523,0.000000,0.741848,0.000000,0.480081,0.000000,0.000000,0.000000,0.000000
1,abundant,0.000000,195,0.768631,0.000003,0.638305,-0.000005,0.575986,-0.000010,0.433534,0.000011,0.741844,-0.000004,0.480070,-0.000011,0.000000,-0.000000,0.000001
2,abundant,0.000000,195,0.768630,0.000003,0.638296,-0.000014,0.575966,-0.000030,0.433556,0.000033,0.741853,0.000005,0.480048,-0.000033,-0.000013,-0.000000,0.000003
3,abundant,0.000001,195,0.768670,0.000042,0.638264,-0.000047,0.575897,-0.000099,0.433634,0.000111,0.741801,-0.000046,0.479972,-0.000110,0.000000,-0.000000,0.000008
4,abundant,0.000003,195,0.768666,0.000038,0.638171,-0.000140,0.575699,-0.000297,0.433856,0.000332,0.741661,-0.000186,0.479752,-0.000329,-0.000000,-0.000001,0.000025
5,abundant,0.000010,195,0.768369,-0.000259,0.637848,-0.000463,0.575011,-0.000985,0.434631,0.001108,0.741016,-0.000832,0.478988,-0.001094,-0.003006,-0.000002,0.000093
6,abundant,0.000030,195,0.768263,-0.000365,0.636948,-0.001362,0.573100,-0.002896,0.436841,0.003318,0.738900,-0.002947,0.476815,-0.003266,-0.005351,-0.000002,0.000326
7,abundant,0.000100,195,0.767630,-0.000998,0.634230,-0.004080,0.567461,-0.008535,0.444259,0.010735,0.728919,-0.012928,0.469769,-0.010313,-0.008300,0.000074,0.001730
8,single_shot,0.000000,195,0.763121,0.000000,0.171743,0.000000,0.383458,0.000000,0.656915,0.000000,0.704522,0.000000,0.502002,0.000000,0.000000,0.000000,0.000000
9,single_shot,0.000000,195,0.763299,0.000179,0.171897,0.000154,0.384954,0.001496,0.656332,-0.000583,0.702649,-0.001873,0.505722,0.003720,-0.000372,0.001569,0.012791


In [4]:
cols = [
    "delta_control",
    "delta_prediction",
    "delta_row_TV",
    "delta_true_edge_mass",
    "delta_edge_AP",
    "delta_centrality_L1",
    "worst_dynamic_control_delta",
    "worst_dynamic_prediction_delta",
    "worst_dynamic_row_TV_delta",
]

parts = []
for regime in ["abundant", "single_shot"]:
    p = summary[summary["regime"] == regime][
        ["regularizer_lambda"] + cols
    ].copy()
    p = p.rename(columns={c: f"{regime}_{c}" for c in cols})
    parts.append(p)

joint = parts[0].merge(
    parts[1],
    on="regularizer_lambda",
    validate="one_to_one",
).sort_values("regularizer_lambda")

joint.to_csv(OUT_DIR / "joint_gini_side_by_side.csv", index=False)

print("Deltas vs no regularization")
display(joint.round(6))


Deltas vs no regularization


,regularizer_lambda,abundant_delta_control,abundant_delta_prediction,abundant_delta_row_TV,abundant_delta_true_edge_mass,abundant_delta_edge_AP,abundant_delta_centrality_L1,abundant_worst_dynamic_control_delta,abundant_worst_dynamic_prediction_delta,abundant_worst_dynamic_row_TV_delta,single_shot_delta_control,single_shot_delta_prediction,single_shot_delta_row_TV,single_shot_delta_true_edge_mass,single_shot_delta_edge_AP,single_shot_delta_centrality_L1,single_shot_worst_dynamic_control_delta,single_shot_worst_dynamic_prediction_delta,single_shot_worst_dynamic_row_TV_delta
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000003,-0.000005,-0.000010,0.000011,-0.000004,-0.000011,0.000000,-0.000000,0.000001,0.000179,0.000154,0.001496,-0.000583,-0.001873,0.003720,-0.000372,0.001569,0.012791
2,0.000000,0.000003,-0.000014,-0.000030,0.000033,0.000005,-0.000033,-0.000013,-0.000000,0.000003,-0.000075,0.003332,0.002129,0.001588,-0.003585,0.009952,-0.002217,0.025951,0.027765
3,0.000001,0.000042,-0.000047,-0.000099,0.000111,-0.000046,-0.000110,0.000000,-0.000000,0.000008,-0.000437,0.006491,0.010175,0.004698,-0.010338,0.034642,-0.004419,0.040329,0.063477
4,0.000003,0.000038,-0.000140,-0.000297,0.000332,-0.000186,-0.000329,-0.000000,-0.000001,0.000025,-0.002209,0.028748,0.028587,0.015218,-0.032544,0.101602,-0.013244,0.124539,0.109603
5,0.000010,-0.000259,-0.000463,-0.000985,0.001108,-0.000832,-0.001094,-0.003006,-0.000002,0.000093,-0.008036,0.088235,0.070088,0.015673,-0.082023,0.239852,-0.025861,0.282586,0.236152
6,0.000030,-0.000365,-0.001362,-0.002896,0.003318,-0.002947,-0.003266,-0.005351,-0.000002,0.000326,-0.023257,0.171664,0.143029,-0.002031,-0.148413,0.417964,-0.044957,0.347238,0.286304
7,0.000100,-0.000998,-0.004080,-0.008535,0.010735,-0.012928,-0.010313,-0.008300,0.000074,0.001730,-0.049886,0.337772,0.265107,-0.077728,-0.230420,0.609086,-0.159192,0.744660,0.594898


In [5]:
print("Low-lambda region")
display(
    by_dyn[
        by_dyn["regularizer_lambda"].isin([1e-7, 3e-7, 1e-6])
    ][
        [
            "regime", "regularizer_lambda", "scenario_class", "dynamics",
            "delta_control", "delta_prediction", "delta_row_TV",
            "delta_true_edge_mass", "delta_edge_AP",
            "delta_centrality_L1",
        ]
    ].round(6)
)

print("\nOutputs:", OUT_DIR)
print("JOINT GINI LOW-REFINEMENT ANALYSIS COMPLETE")


Low-lambda region


,regime,regularizer_lambda,scenario_class,dynamics,delta_control,delta_prediction,delta_row_TV,delta_true_edge_mass,delta_edge_AP,delta_centrality_L1
9,abundant,0.000000,structured_mechanism,friedkinjohnsen,0.000000,-0.000001,-0.000001,0.000001,0.000000,-0.000001
10,abundant,0.000000,structured_mechanism,hegselmannkrause,0.000000,-0.000000,-0.000005,0.000011,0.000003,-0.000009
11,abundant,0.000000,structured_mechanism,nonlinearinfluence,0.000000,-0.000006,-0.000010,0.000011,0.000000,-0.000018
12,abundant,0.000000,unstructured_random,coca,0.000023,-0.000005,-0.000014,0.000015,-0.000000,-0.000012
13,abundant,0.000000,unstructured_random,friedkinjohnsen,0.000000,-0.000000,-0.000001,0.000001,0.000000,-0.000001
14,abundant,0.000000,unstructured_random,hegselmannkrause,0.000000,-0.000004,-0.000009,0.000010,0.000000,-0.000011
15,abundant,0.000000,unstructured_random,laplacian,0.000000,-0.000016,-0.000032,0.000033,0.000000,-0.000031
16,abundant,0.000000,unstructured_random,nonlinearinfluence,0.000000,-0.000007,-0.000013,0.000013,-0.000030,-0.000012
17,abundant,0.000000,unstructured_random,repulsion,0.000000,-0.000000,0.000001,-0.000001,0.000000,-0.000002
18,abundant,0.000000,structured_mechanism,friedkinjohnsen,0.000000,-0.000002,-0.000002,0.000003,0.000000,-0.000004



Outputs: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\joint_gini_sweep_with_low_refinement_2026_09_29
JOINT GINI LOW-REFINEMENT ANALYSIS COMPLETE
